In [23]:
import numpy as np
import matplotlib.pyplot as plt

import scipy
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

In [17]:
# Taken from Vercer PDE for Asian Options

def q_continuous_tau(tau, r, T):
    """
    Computes q as a function of tau = T - t.

    Parameters
    ----------
    tau : float
        Forward time variable tau = T - t.
    r : float
        Risk-free interest rate.
    T : float
        Maturity.

    Returns
    -------
    float
        The value q(tau) = (1 - exp(-r tau)) / (r T),
        with the correct limiting formula when r is close to zero.
    """
    if abs(r) < 1e-12:
        return tau / T
    return (1.0 - np.exp(-r * tau)) / (r * T)

def price_asian_vecer_fixed_strike(
    S0,
    K,
    r,
    sigma,
    T,
    option_type="call",
    z_min=-1.5,
    z_max=1.5,
    Nz=801,
    Nt=1000,
    return_grid=False
):
    """
    Prices a continuously sampled fixed-strike Asian option using the one-dimensional Věčer PDE.

    Parameters
    ----------
    S0 : float
        Initial stock price.
    K : float
        Strike price.
    r : float
        Continuously compounded risk-free rate.
    sigma : float
        Volatility.
    T : float
        Maturity in years.
    option_type : str
        Either "call" or "put".
    z_min : float
        Lower bound of the spatial z-grid.
    z_max : float
        Upper bound of the spatial z-grid.
    Nz : int
        Number of z-grid points.
    Nt : int
        Number of time steps.
    return_grid : bool
        If True, return the full z-grid and final PDE solution.

    Returns
    -------
    price : float
        The Asian option price.
    z0 : float
        The transformed initial state.
    z : numpy.ndarray
        The z-grid, returned only if return_grid=True.
    v : numpy.ndarray
        The PDE solution at tau=T, returned only if return_grid=True.
    """

    # ------------------------------------------------------------
    # 1. Build the spatial grid in z.
    # ------------------------------------------------------------
    z = np.linspace(z_min, z_max, Nz)
    dz = z[1] - z[0]
    dt = T / Nt

    # ------------------------------------------------------------
    # 2. Initial condition in tau.
    # ------------------------------------------------------------
    option_type = option_type.lower()

    if option_type == "call":
        v = np.maximum(z, 0.0)
    elif option_type == "put":
        v = np.maximum(-z, 0.0)
    else:
        raise ValueError("option_type must be either 'call' or 'put'.")

    # ------------------------------------------------------------
    # 3. Boundary conditions.
    #
    # For a call:
    #   as z -> -infinity, v -> 0
    #   as z -> +infinity, v ~ z
    #
    # For a put:
    #   as z -> -infinity, v ~ -z
    #   as z -> +infinity, v -> 0
    # ------------------------------------------------------------
    if option_type == "call":
        left_boundary = 0.0
        right_boundary = z_max
    else:
        left_boundary = -z_min
        right_boundary = 0.0

    # ------------------------------------------------------------
    # 4. Time stepping.
    #
    # At each step we solve:
    #
    #   v_new - dt * a * Dzz(v_new) = v_old
    # ------------------------------------------------------------
    for n in range(Nt):
        tau_next = (n + 1) * dt

        q = q_continuous_tau(tau_next, r, T)

        # Interior grid points only.
        z_interior = z[1:-1]

        # Diffusion coefficient a(tau,z).
        a = 0.5 * sigma**2 * (q - z_interior)**2

        # lambda_j = dt * a_j / dz^2
        lam = dt * a / dz**2

        # Tridiagonal matrix:
        #
        # -lam_j * v_{j-1}^{new}
        # + (1 + 2 lam_j) * v_j^{new}
        # - lam_j * v_{j+1}^{new}
        # = v_j^{old}
        lower_diagonal = -lam[1:]
        main_diagonal = 1.0 + 2.0 * lam
        upper_diagonal = -lam[:-1]

        A = diags(
            diagonals=[lower_diagonal, main_diagonal, upper_diagonal],
            offsets=[-1, 0, 1],
            format="csc"
        )

        rhs = v[1:-1].copy()

        # Add boundary-condition contributions.
        rhs[0] += lam[0] * left_boundary
        rhs[-1] += lam[-1] * right_boundary

        # Solve for the new interior values.
        v_new_interior = spsolve(A, rhs)

        # Reconstruct the full solution including boundaries.
        v[0] = left_boundary
        v[-1] = right_boundary
        v[1:-1] = v_new_interior

    # ------------------------------------------------------------
    # 5. Compute the initial transformed state z0.
    #
    # At t=0 it holds that
    #
    # z0 = q(0) - exp(-rT) K/S0.
    #
    # In tau variables, q(0) means q_tau(T).
    # ------------------------------------------------------------
    q0 = q_continuous_tau(T, r, T)
    z0 = q0 - np.exp(-r * T) * K / S0

    # Interpolate the PDE solution at z0.
    v_at_z0 = np.interp(z0, z, v)

    # Recover the actual option price.
    price = S0 * v_at_z0

    if return_grid:
        return price, z0, z, v

    return price

In [2]:
# taken from Asian_Options
def GBM_paths(S0, sigma, t, r, mu, n_sims, n_steps):
    """Simulates stock paths as geometric Brownian Motions
    Inputs:
    S0 (float): Underlying stock price at time 0
    sigma (float): Yearly volatility
    t (float): Time to expiration (years)
    r (float): Risk-free interest rate
    mu (float): Drift of log-returns
    n_sims (int): Number of simulated paths
    n_steps (int): Number of steps in each simulated path, each step interval has length t/n_steps
    
    Return (np.array): Array of stock paths
    """
    
    dt = t/n_steps
    noise = np.random.normal(loc = 0, scale = 1, size = (n_sims, n_steps))
    log_returns = (mu+r-sigma**2*(0.5))*dt + sigma*np.sqrt(dt)*noise
    exponent = np.cumsum(log_returns, axis = 1)
    paths = S0*np.exp(exponent)
    paths_with_start = np.insert(paths, 0, S0, axis = 1)

    return paths_with_start

def asian_call_price_mc(S0, K,sigma,t,r,n_sims = 1000,n_steps = 2520): #Function that returns a 1x2 array. 
                                                                        #One entry is the Monte-Carlo estimate of the price of the Asian Call.
                                                                        #The other entry is the standard error of the estimate. 
    stock_paths = GBM_paths(S0, sigma, t, r, 0, n_sims, n_steps)
    
    avg_val_of_stocks = np.mean(stock_paths, axis = 1) #mean of each stock path
    
    payoffs = np.maximum(avg_val_of_stocks - K, 0) # Call payoffs
    
    prices = np.exp(-r*t) * payoffs #discounting to time 0
    
    return [np.average(prices), np.std(prices)/np.sqrt(n_sims)] # Actual estimate is zero-th entry of the resulting array
                                                                # The standard error of the estimate is the first entry 



In [20]:
# Taken from Volatility.ipynb

def theoretical_payout(K, r, T, sigma, Type = 'call', Greeks = False):
    """
Prices a European option using the Black-Scholes equation, optinally returning the Greeks as well.
Note: This is normalized so that S_0=1. I.e. the return value and K are in terms of returns. This also affects the Greeks

Parameters
----------
K: float
    Strike price
r: float
    Risk-free rate (continuously compounded)
T: float
    Time to contract maturity
sigma: float
    Volatility
Type: str
    Type of contract; either 'call' or 'put'
Greeks: bool
    Whether or not to also return a dict of the Greeks for the contract
"""
    assert Type in ['call', 'put'], "Type of contract must either be a call or a put"

    # d_1 and d_2 are called dplus and dminus in the code respectively
    dplus  = 1/(sigma*np.sqrt(T))*(np.log(1/K)+(r+(sigma**2)/2)*T)
    dminus = 1/(sigma*np.sqrt(T))*(np.log(1/K)+(r-(sigma**2)/2)*T)
    
    if Type == 'call':
        value = scipy.stats.norm.cdf(dplus) - K*np.exp(-r*T)*scipy.stats.norm.cdf(dminus)
    else: # This is the value for a put
        value = -scipy.stats.norm.cdf(-dplus) + K*np.exp(-r*T)*scipy.stats.norm.cdf(-dminus)

    if not Greeks:
        return value

    greeks = {
        'delta': scipy.stats.norm.cdf(dplus) if Type == 'call' else scipy.stats.norm.cdf(dplus) - 1,
        'gamma': scipy.stats.norm.pdf(dplus) / (sigma * np.sqrt(T)),
        'vega': scipy.stats.norm.pdf(dplus) * np.sqrt(T),
        'theta': -scipy.stats.norm.pdf(dplus) * sigma / (2*np.sqrt(T)) - r * K * np.exp(-r*T) * scipy.stats.norm.cdf(dminus) if Type == 'call' else \
                 -scipy.stats.norm.pdf(dplus) * sigma / (2*np.sqrt(T)) + r * K * np.exp(-r*T) * scipy.stats.norm.cdf(-dminus),
        'rho': K * T * np.exp(-r*T) * scipy.stats.norm.cdf(dminus) if Type == 'call' else \
               -K * T * np.exp(-r*T) * scipy.stats.norm.cdf(-dminus)
    }

    return value, greeks

In [29]:
S0 = 240
sigma = .8
T = 1
K = 280
r = 0.035

In [ ]:
mean, SE = asian_call_price_mc(S0, K, sigma, T, r, n_sims = 10000)
print(f'95% Confidence interval given by ({mean-2*SE}, {mean+2*SE})')

In [27]:
price_asian_vecer_fixed_strike(S0, K, r, sigma, T)

np.float64(40.89630469069131)

In [28]:
theoretical_payout(K / S0, r, T, sigma) * S0

np.float64(74.1214376933455)

In [21]:
help(theoretical_payout)

Help on function theoretical_payout in module __main__:

theoretical_payout(K, r, T, sigma, Type='call', Greeks=False)
    Prices a European option using the Black-Scholes equation, optinally returning the Greeks as well.
    Note: This is normalized so that S_0=1. I.e. the return value and K are in terms of returns. This also affects the Greeks

    Parameters
    ----------
    K: float
        Strike price
    r: float
        Risk-free rate (continuously compounded)
    T: float
        Time to contract maturity
    sigma: float
        Volatility
    Type: str
        Type of contract; either 'call' or 'put'
    Greeks: bool
        Whether or not to also return a dict of the Greeks for the contract

